In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss


In [ ]:

# Load the training data
train_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/cirrhosis_patient/train.csv'
train_df = pd.read_csv(train_data_path)

# Display the first few rows of the dataset to understand its structure
print(train_df.head())


     id  N_Days             Drug    Age  ... Platelets Prothrombin Stage Status
0  6703    1153          Placebo  14772  ...     236.0         9.9   3.0     CL
1  5815    1447          Placebo  14754  ...     306.0         9.5   2.0      C
2  3429    2891          Placebo  14899  ...     322.0         9.5   2.0      C
3  2405     334  D-penicillamine  22369  ...     156.0        11.0   2.0      C
4  1410    3820          Placebo  20597  ...     119.0        11.7   4.0      D

[5 rows x 20 columns]


In [ ]:


# Check for missing values
print(train_df.isnull().sum())

# Handle missing values (e.g., fill with mean for numerical columns)
numerical_cols = train_df.select_dtypes(include=[np.number]).columns
train_df[numerical_cols] = train_df[numerical_cols].fillna(train_df[numerical_cols].mean())

# Handle missing values for categorical columns (e.g., fill with mode)
categorical_cols = train_df.select_dtypes(include=[object]).columns
train_df[categorical_cols] = train_df[categorical_cols].fillna(train_df[categorical_cols].mode().iloc[0])

# Encode categorical variables
train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)

# Split the data into features (X) and target (y)
X = train_df.drop(columns=['Status'])
y = train_df['Status']

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize numerical features
scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_val[numerical_cols] = scaler.transform(X_val[numerical_cols])



id               0
N_Days           0
Drug             0
Age              0
Sex              0
Ascites          0
Hepatomegaly     0
Spiders          0
Edema            0
Bilirubin        0
Cholesterol      0
Albumin          0
Copper           0
Alk_Phos         0
SGOT             0
Tryglicerides    0
Platelets        0
Prothrombin      0
Stage            0
Status           0
dtype: int64
---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
Cell In[4], line 32
     30 pass
     31 print('##active_line16##')
---> 32 X = train_df.drop(columns=['Status'])
     33 print('##active_line17##')
     34 y = train_df['Status']

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\pandas\core\frame.py:5581, in DataFrame.drop(self, labels, axis, index, columns, level, inplace, errors)
   5433 def drop(
   5434     self,
   5435     labels: IndexLabel | None = None,

In [ ]:


# Check the actual column names
print(train_df.columns)

# Ensure the 'Status' column is correctly handled
# If the 'Status' column is not present, check for alternative names like 'status', 'Status_', etc.
if 'Status' not in train_df.columns:
    # Replace 'Status' with the actual column name if necessary
    correct_status_column = None
    for col in train_df.columns:
        if 'status' in col.lower():
            correct_status_column = col
            break

    if correct_status_column:
        print(f"Using '{correct_status_column}' as the status column.")
        X = train_df.drop(columns=[correct_status_column])
        y = train_df[correct_status_column]
    else:
        raise Exception("Status column not found in the dataset.")
else:
    X = train_df.drop(columns=['Status'])
    y = train_df['Status']

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize numerical features
scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_val[numerical_cols] = scaler.transform(X_val[numerical_cols])




Index(['id', 'N_Days', 'Age', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper',
       'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin',
       'Stage', 'Drug_Placebo', 'Sex_M', 'Ascites_Y', 'Hepatomegaly_Y',
       'Spiders_Y', 'Edema_S', 'Edema_Y', 'Status_CL', 'Status_D'],
      dtype='object')
Using 'Status_CL' as the status column.


In [ ]:



# Initialize the RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model
model.fit(X_train, y_train)

# Make predictions on the validation set
y_pred_proba = model.predict_proba(X_val)

# Calculate the log loss
log_loss_value = log_loss(y_val, y_pred_proba)

# Print the log loss
print(f"Log Loss: {log_loss_value}")





Log Loss: 0.15577501315505182
